# Clase 3 · Preparación de datos paso a paso

En esta notebook vamos a ver **una función por vez**, con tablas pequeñas y resultados fáciles de comparar.

En cada ejemplo responderemos: **¿qué hace?, ¿cuándo conviene usarla?, ¿cuándo no conviene?, ¿qué cambió?**

Abrí la notebook en Jupyter o Google Colab. Ejecutá primero la celda de importaciones. Después podés ejecutar cada ejemplo por separado: cada uno crea sus propios datos. No necesitás descargar archivos adicionales.

**No hay que aplicar todas las funciones a los mismos datos.** Son herramientas para necesidades diferentes.

## Antes de comenzar

Una tabla tiene **filas** (personas, productos u observaciones) y **columnas** (edad, precio, ciudad, etc.).

- **Numérica:** expresa una cantidad, como ingreso o número de compras.
- **Categórica nominal:** tiene categorías sin orden, como ciudad.
- **Categórica ordinal:** tiene categorías ordenadas, como bajo, medio y alto.
- **Categórica binaria:** tiene dos categorías, como sí y no.

Algunas técnicas comparan distancias o realizan cálculos sensibles a las unidades: un ingreso en pesos puede dominar numéricamente a una edad en años. Escalar ayuda a evitar ese desequilibrio. Otros métodos, como los árboles, suelen no necesitarlo. Estudiaremos esos modelos más adelante.

### Importaciones
`pandas` permite crear tablas y `numpy` representar valores faltantes. Las demás herramientas se importan junto a su ejemplo.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.precision", 3)

**Ayuda para leer el código:** `datos[["edad"]]` selecciona una columna manteniendo formato de tabla. Muchas funciones de scikit-learn esperan ese formato. `fit_transform()` calcula lo necesario y transforma el pequeño ejemplo. `.ravel()` acomoda el resultado en una sola columna para mostrarlo.

Si te falta alguna biblioteca en tu computadora, instalá pandas, numpy y scikit-learn. Para `OneHotEncoder` usamos scikit-learn 1.2 o posterior.

## 1. Mirar los datos antes de modificarlos

**¿Qué hace?** `info()` muestra los tipos; `isna().sum()` cuenta faltantes; `value_counts()` cuenta categorías.

**Cuándo usarla:** al comenzar a trabajar con una tabla, para identificar qué necesita preparación.

**Cuándo no usarla o tener cuidado:** no tomes decisiones solo por el tipo de almacenamiento: un código postal numérico puede representar una categoría.

Vamos a probarlo con pocos datos:

In [ ]:
datos = pd.DataFrame({
    "edad": [20, 30, np.nan, 40],
    "ciudad": ["Ushuaia", "Tolhuin", "Ushuaia", "Tolhuin"]
})
display(datos)
datos.info()
print("Faltantes por columna:")
display(datos.isna().sum())
print("Cantidad de personas por ciudad:")
display(datos["ciudad"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   edad    3 non-null      float64
 1   ciudad  4 non-null      object 
dtypes: float64(1), object(1)
memory usage: 196.0+ bytes
Faltantes por columna:
Cantidad de personas por ciudad:


,edad,ciudad
0,20.0,Ushuaia
1,30.0,Tolhuin
2,NaN,Ushuaia
3,40.0,Tolhuin


edad      1
ciudad    0
dtype: int64

ciudad
Ushuaia    2
Tolhuin    2
Name: count, dtype: int64

**Cómo leer el resultado:** falta una edad. Cada ciudad aparece dos veces. Inspeccionar permite detectar problemas, pero no los corrige automáticamente.

## 2. Convertir texto a números: pd.to_numeric()

**¿Qué hace?** convierte textos como `"100"` en números. Con `errors="coerce"`, un texto inválido se convierte en `NaN` (faltante).

**Cuándo usarla:** cuando una columna representa cantidades, pero fue importada como texto.

**Cuándo no usarla o tener cuidado:** no la uses para convertir ciudades o nombres en números. Revisá separadores decimales y de miles antes de convertir.

Vamos a probarlo con pocos datos:

In [ ]:
datos = pd.DataFrame({"precio_texto": ["100", "200", "sin dato", "400"]})
datos["precio_numerico"] = pd.to_numeric(datos["precio_texto"], errors="coerce")
display(datos)

,precio_texto,precio_numerico
0,100,100.0
1,200,200.0
2,sin dato,NaN
3,400,400.0


**Cómo leer el resultado:** los precios válidos se recuperaron. "sin dato" quedó como faltante: no se inventó un precio ni se reemplazó por cero.

## 3. Unificar categorías: strip(), lower() y replace()

**¿Qué hace?** quita espacios externos, pasa a minúsculas y reemplaza equivalencias que conocemos.

**Cuándo usarla:** cuando distintas escrituras representan la misma categoría.

**Cuándo no usarla o tener cuidado:** no unas categorías que tienen distinto significado. La limpieza requiere conocer los datos.

Vamos a probarlo con pocos datos:

In [ ]:
datos = pd.DataFrame({"canal_original": [" WEB ", "web", "Local", "telef."]})
datos["canal_limpio"] = datos["canal_original"].str.strip()
datos["canal_limpio"] = datos["canal_limpio"].str.lower()
datos["canal_limpio"] = datos["canal_limpio"].replace({"telef.": "teléfono"})
display(datos)

,canal_original,canal_limpio
0,WEB,web
1,web,web
2,Local,local
3,telef.,teléfono


**Cómo leer el resultado:** " WEB " y "web" ahora son una sola categoría. "telef." se reemplazó por una etiqueta completa.

## 4. Quitar repeticiones accidentales: drop_duplicates()

**¿Qué hace?** elimina filas repetidas y conserva una copia.

**Cuándo usarla:** cuando confirmamos que una observación se cargó dos veces por error.

**Cuándo no usarla o tener cuidado:** no la apliques automáticamente a operaciones reales: dos compras iguales pueden ser dos compras válidas.

Vamos a probarlo con pocos datos:

In [ ]:
datos = pd.DataFrame({"id_persona": [1, 2, 2], "edad": [20, 30, 30]})
print("Antes:")
display(datos)
print("Después:")
display(datos.drop_duplicates())

Antes:
Después:


,id_persona,edad
0,1,20
1,2,30
2,2,30


,id_persona,edad
0,1,20
1,2,30


**Cómo leer el resultado:** la persona 2 figuraba dos veces con los mismos datos. En este ejemplo sabemos que fue un error de carga.

## 5. Completar números faltantes: SimpleImputer()

**¿Qué hace?** reemplaza faltantes por un valor calculado, por ejemplo la media o la mediana.

**Cuándo usarla:** cuando necesitás conservar una fila y resulta razonable estimar su dato faltante. La mediana es menos sensible a extremos.

**Cuándo no usarla o tener cuidado:** no lo uses para ocultar errores sin investigarlos ni para inventar la respuesta que querés predecir. Completar datos modifica su distribución.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.impute import SimpleImputer

datos = pd.DataFrame({"importe": [10, 20, 30, 1000, np.nan]})
media = SimpleImputer(strategy="mean")
mediana = SimpleImputer(strategy="median")
datos["con_media"] = media.fit_transform(datos[["importe"]]).ravel()
datos["con_mediana"] = mediana.fit_transform(datos[["importe"]]).ravel()
display(datos)

,importe,con_media,con_mediana
0,10.0,10.0,10.0
1,20.0,20.0,20.0
2,30.0,30.0,30.0
3,1000.0,1000.0,1000.0
4,NaN,265.0,25.0


**Cómo leer el resultado:** el faltante se completa con 265 usando la media y con 25 usando la mediana. El 1000 afecta mucho más a la media. Ninguna opción recupera el importe verdadero.

## 6. Completar categorías faltantes: SimpleImputer()

**¿Qué hace?** puede usar la categoría más frecuente (`most_frequent`) o una etiqueta fija (`constant`).

**Cuándo usarla:** cuando es adecuado sustituir una categoría desconocida por la más común o hacer visible la ausencia como "sin dato".

**Cuándo no usarla o tener cuidado:** no uses la moda si reforzar la categoría mayoritaria distorsiona el problema. "Sin dato" tampoco representa una categoría real conocida.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.impute import SimpleImputer

datos = pd.DataFrame({"ciudad": ["Ushuaia", "Tolhuin", "Ushuaia", np.nan]})
moda = SimpleImputer(strategy="most_frequent")
etiqueta = SimpleImputer(strategy="constant", fill_value="sin dato")
datos["con_moda"] = moda.fit_transform(datos[["ciudad"]]).ravel()
datos["con_etiqueta"] = etiqueta.fit_transform(datos[["ciudad"]]).ravel()
display(datos)

,ciudad,con_moda,con_etiqueta
0,Ushuaia,Ushuaia,Ushuaia
1,Tolhuin,Tolhuin,Tolhuin
2,Ushuaia,Ushuaia,Ushuaia
3,NaN,Ushuaia,sin dato


**Cómo leer el resultado:** la moda coloca Ushuaia. La etiqueta fija muestra "sin dato" y deja explícito que desconocemos la ciudad.

## 7. Llevar una columna a 0–1: MinMaxScaler()

**¿Qué hace?** resta el mínimo y divide por la diferencia entre máximo y mínimo. Trabaja por columna.

**Cuándo usarla:** cuando interesa una referencia común de 0 a 1 y el procedimiento que usarás necesita controlar escalas.

**Cuándo no usarla o tener cuidado:** no lo uses como solución a valores extremos: pueden comprimir al resto. Tampoco hace falta escalar identificadores o códigos de categorías.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import MinMaxScaler

datos = pd.DataFrame({"precio": [100, 200, 300]})
escalador = MinMaxScaler()
datos["precio_0_a_1"] = escalador.fit_transform(datos[["precio"]]).ravel()
display(datos)

,precio,precio_0_a_1
0,100,0.0
1,200,0.5
2,300,1.0


**Cómo leer el resultado:** 100 es el mínimo y queda en 0; 300 es el máximo y queda en 1; 200 queda en 0.5. Se conserva el orden. No son probabilidades. Si luego se aplica la misma escala a un valor superior al máximo, puede superar 1.

## 8. Estandarizar: StandardScaler()

**¿Qué hace?** resta la media y divide por la desviación estándar de cada columna.

**Cuándo usarla:** cuando un procedimiento sensible a las escalas necesita comparar variables con unidades muy diferentes.

**Cuándo no usarla o tener cuidado:** no lo uses para "volver normales" los datos: no cambia la forma de su distribución. Es sensible a extremos y no garantiza valores entre 0 y 1.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import StandardScaler

datos = pd.DataFrame({"puntaje": [10, 20, 30]})
escalador = StandardScaler()
datos["estandarizado"] = escalador.fit_transform(datos[["puntaje"]]).ravel()
display(datos)
print("Media:", round(datos["estandarizado"].mean(), 3))
print("Desviación:", round(datos["estandarizado"].std(ddof=0), 3))

Media: 0.0
Desviación: 1.0


,puntaje,estandarizado
0,10,-1.225
1,20,0.000
2,30,1.225


**Cómo leer el resultado:** 20 coincide con la media y queda en 0. Los valores menores quedan negativos y los mayores positivos. La desviación es 1 para esta columna no constante; `ddof=0` coincide con el cálculo del escalador.

## 9. Solo centrar: StandardScaler(with_std=False)

**¿Qué hace?** resta la media sin dividir por la desviación.

**Cuándo usarla:** cuando necesitás expresar diferencias respecto del promedio y conservar las unidades originales.

**Cuándo no usarla o tener cuidado:** no alcanza si necesitás igualar escalas: centrar ingresos y edades no elimina la diferencia de unidades.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import StandardScaler

datos = pd.DataFrame({"puntaje": [10, 20, 30]})
centrador = StandardScaler(with_std=False)
datos["centrado"] = centrador.fit_transform(datos[["puntaje"]]).ravel()
display(datos)

,puntaje,centrado
0,10,-10.0
1,20,0.0
2,30,10.0


**Cómo leer el resultado:** la media original es 20. Al restarla obtenemos -10, 0 y 10. Centrar y estandarizar no son exactamente lo mismo.

## 10. Escalar con menor influencia de extremos: RobustScaler()

**¿Qué hace?** resta la mediana y divide por el rango intercuartílico, que mide la dispersión de la parte central.

**Cuándo usarla:** cuando hay valores extremos válidos y necesitás que influyan menos en los parámetros de escala.

**Cuándo no usarla o tener cuidado:** no lo uses para corregir errores de carga o eliminar extremos: los valores extremos siguen presentes.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import RobustScaler

datos = pd.DataFrame({"importe": [10, 11, 12, 13, 1000]})
escalador = RobustScaler()
datos["escalado_robusto"] = escalador.fit_transform(datos[["importe"]]).ravel()
display(datos)

,importe,escalado_robusto
0,10,-1.0
1,11,-0.5
2,12,0.0
3,13,0.5
4,1000,494.0


**Cómo leer el resultado:** 12 es la mediana y queda en 0. Los valores centrales quedan cerca de cero; 1000 sigue siendo extremo. No se garantiza un rango de 0 a 1.

**¿Cuál elegir para un modelo sensible a escalas?**

**StandardScaler()**: cuando interesa que las variables tengan una dispersión comparable. Es un punto de partida habitual para modelos sensibles a distancias, regularización u optimización.
**MinMaxScaler()**: cuando interesa trabajar con un rango de referencia común, como 0–1, y los mínimos y máximos son representativos.

Si hay valores extremos importantes, ambos pueden verse afectados; conviene analizar esos valores y considerar una alternativa como **RobustScaler()**.

La elección puede cambiar cuánto pesa cada variable en una distancia: StandardScaler iguala la desviación estándar; MinMaxScaler iguala el rango. Por eso, aunque ambos escalan, no necesariamente producen el mismo comportamiento del modelo.

## 11. Normalizar filas: normalize() y Normalizer()

**¿Qué hace?** divide cada fila por una medida de su tamaño, llamada norma. Permite comparar perfiles.

**Cuándo usarla:** cuando interesa la proporción entre componentes y no el volumen total. Usá componentes con unidades comparables, como conteos de interacciones.

**Cuándo no usarla o tener cuidado:** no lo uses si el volumen total es importante. Tampoco mezcles edad e ingreso en una fila sin justificarlo: las unidades afectan el perfil.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import normalize

# Dos personas con iguales proporciones, pero distinto volumen.
datos = pd.DataFrame({"clics_A": [3, 30], "clics_B": [4, 40]})
print("Original:")
display(datos)
print("Normalización L1:")
display(pd.DataFrame(normalize(datos, norm="l1"), columns=datos.columns))
print("Normalización L2:")
display(pd.DataFrame(normalize(datos, norm="l2"), columns=datos.columns))
print("Normalización por máximo:")
display(pd.DataFrame(normalize(datos, norm="max"), columns=datos.columns))

Original:
Normalización L1:
Normalización L2:
Normalización por máximo:


,clics_A,clics_B
0,3,4
1,30,40


,clics_A,clics_B
0,0.429,0.571
1,0.429,0.571


,clics_A,clics_B
0,0.6,0.8
1,0.6,0.8


,clics_A,clics_B
0,0.75,1.0
1,0.75,1.0


**Cómo leer el resultado:** ambas personas quedan iguales. L1 divide por 7 y 70: con estos datos positivos cada fila suma 1. L2 divide por 5 y 50: la longitud es 1, no la suma. "max" divide por 4 y 40. Se perdió la diferencia de volumen.

`Normalizer()` permite hacer la misma operación mediante un objeto. Elegí una de las dos formas; no hace falta aplicarlas una después de la otra. Con valores negativos, L1 hace que **los valores absolutos** sumen 1. Una fila de ceros sigue en cero.

In [ ]:
from sklearn.preprocessing import Normalizer

datos = pd.DataFrame({"clics_A": [3, 30], "clics_B": [4, 40]})
normalizador = Normalizer(norm="l2")
resultado = normalizador.fit_transform(datos)
display(pd.DataFrame(resultado, columns=datos.columns))

,clics_A,clics_B
0,0.6,0.8
1,0.6,0.8


## 12. Convertir cantidades en indicadores: Binarizer()

**¿Qué hace?** devuelve 1 si el valor supera un umbral y 0 si es menor o igual.

**Cuándo usarla:** cuando importa presencia/ausencia o superar una condición, por ejemplo haber realizado alguna compra.

**Cuándo no usarla o tener cuidado:** no lo uses si necesitás conservar la cantidad: 1 compra y 20 compras pueden quedar iguales.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import Binarizer

datos = pd.DataFrame({"compras": [0, 1, 3, 4, 20]})
datos["alguna_compra"] = Binarizer(threshold=0).fit_transform(datos[["compras"]]).ravel()
datos["mas_de_3"] = Binarizer(threshold=3).fit_transform(datos[["compras"]]).ravel()
display(datos)

,compras,alguna_compra,mas_de_3
0,0,0,0
1,1,1,0
2,3,1,0
3,4,1,1
4,20,1,1


**Cómo leer el resultado:** con umbral 3, el valor 3 produce 0 y el 4 produce 1. La condición es **mayor que**, no mayor o igual. Binarizar una entrada no determina si la tarea es regresión o clasificación.

## 13. Categorías con orden: OrdinalEncoder()

**¿Qué hace?** asigna números respetando un orden que indicamos explícitamente.

**Cuándo usarla:** cuando hay una jerarquía real, como bajo, medio y alto.

**Cuándo no usarla o tener cuidado:** no lo uses como escala numérica para categorías sin orden, como ciudades. Incluso con orden, los códigos no prueban que los saltos sean iguales.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

datos = pd.DataFrame({"nivel": ["medio", "bajo", "alto", "medio"]})
codificador = OrdinalEncoder(categories=[["bajo", "medio", "alto"]])
datos["nivel_numerico"] = codificador.fit_transform(datos[["nivel"]]).ravel()
display(datos)

,nivel,nivel_numerico
0,medio,1.0
1,bajo,0.0
2,alto,2.0
3,medio,1.0


**Cómo leer el resultado:** bajo = 0, medio = 1 y alto = 2. Definimos el orden nosotros. No significa que pasar de bajo a medio tenga el mismo efecto que pasar de medio a alto.

## 14. Categorías sin orden: OneHotEncoder()

**¿Qué hace?** crea una columna indicadora para cada categoría.

**Cuándo usarla:** cuando una variable nominal debe representarse numéricamente sin inventar una jerarquía.

**Cuándo no usarla o tener cuidado:** no lo apliques a una cantidad continua con muchos valores distintos. Con muchas categorías puede generar demasiadas columnas. No es necesario si la implementación elegida ya trata las categorías directamente.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

datos = pd.DataFrame({"ciudad": ["Ushuaia", "Tolhuin", "Río Grande", "Ushuaia"]})
codificador = OneHotEncoder(sparse_output=False)
resultado = codificador.fit_transform(datos[["ciudad"]])
nombres = codificador.get_feature_names_out(["ciudad"])
indicadores = pd.DataFrame(resultado, columns=nombres)
display(pd.concat([datos, indicadores], axis=1))

,ciudad,ciudad_Río Grande,ciudad_Tolhuin,ciudad_Ushuaia
0,Ushuaia,0.0,0.0,1.0
1,Tolhuin,0.0,1.0,0.0
2,Río Grande,1.0,0.0,0.0
3,Ushuaia,0.0,0.0,1.0


**Cómo leer el resultado:** cada fila tiene 1 en su ciudad y 0 en las demás. Ninguna ciudad se considera "mayor" que otra. Los nombres de salida muestran a qué categoría pertenece cada columna.

## 15. Otra forma de one-hot: pd.get_dummies()

**¿Qué hace?** genera los indicadores directamente con pandas.

**Cuándo usarla:** cuando querés explorar o convertir una tabla pequeña de forma sencilla.

**Cuándo no usarla o tener cuidado:** no vuelvas a codificar los indicadores obtenidos con OneHotEncoder. Con muchas categorías también crece el número de columnas; el esquema depende de las categorías presentes en la tabla.

Vamos a probarlo con pocos datos:

In [ ]:
datos = pd.DataFrame({"color": ["rojo", "azul", "verde", "rojo"]})
indicadores = pd.get_dummies(datos["color"], prefix="color", dtype=int)
display(pd.concat([datos, indicadores], axis=1))

,color,color_azul,color_rojo,color_verde
0,rojo,0,1,0
1,azul,1,0,0
2,verde,0,0,1
3,rojo,0,1,0


**Cómo leer el resultado:** cada color tiene su columna. Es una alternativa a OneHotEncoder para esta demostración, no un paso adicional obligatorio.

## 16. Dos categorías: map()

**¿Qué hace?** aplica una correspondencia explícita, por ejemplo no = 0 y sí = 1.

**Cuándo usarla:** cuando hay dos categorías conocidas y querés una columna fácil de interpretar. También permite otros mapeos definidos por vos.

**Cuándo no usarla o tener cuidado:** no asignes una escala numérica arbitraria a varias categorías nominales. Una etiqueta que no aparece en el diccionario queda como faltante.

Vamos a probarlo con pocos datos:

In [ ]:
datos = pd.DataFrame({"tiene_app": ["sí", "no", "sí", "sin respuesta"]})
datos["codigo"] = datos["tiene_app"].map({"no": 0, "sí": 1})
display(datos)

,tiene_app,codigo
0,sí,1.0
1,no,0.0
2,sí,1.0
3,sin respuesta,NaN


**Cómo leer el resultado:** "sin respuesta" queda como NaN. No se debe confundir desconocimiento con "no"; requiere una decisión aparte.

## 17. Codificar la respuesta de clasificación: LabelEncoder()

**¿Qué hace?** convierte las etiquetas de la variable objetivo en identificadores numéricos y permite recuperar sus nombres.

**Cuándo usarla:** cuando la respuesta que querés predecir es una categoría y la herramienta necesita códigos. Muchos clasificadores aceptan texto directamente, así que no siempre hace falta.

**Cuándo no usarla o tener cuidado:** no lo uses para escalar una cantidad que querés predecir ni como solución general para ciudades u otras columnas de entrada sin orden.

Vamos a probarlo con pocos datos:

In [ ]:
from sklearn.preprocessing import LabelEncoder

respuestas = pd.Series(["renueva", "no renueva", "renueva"])
codificador = LabelEncoder()
codigos = codificador.fit_transform(respuestas)
display(pd.DataFrame({"respuesta": respuestas, "codigo": codigos}))
print("Recuperamos las etiquetas:")
print(codificador.inverse_transform(codigos))

Recuperamos las etiquetas:
['renueva' 'no renueva' 'renueva']


,respuesta,codigo
0,renueva,1
1,no renueva,0
2,renueva,1


**Cómo leer el resultado:** los números identifican clases; no son cantidades ni se estandarizan. Si predecimos gasto en pesos estamos en regresión; si predecimos renueva/no renueva estamos en clasificación. En ambos casos las entradas pueden contener números y categorías.

## Resumen: ¿qué necesito hacer?

| Necesidad | Función | Principal precaución |
|---|---|---|
| Ver faltantes y tipos | `info`, `isna`, `value_counts` | Inspeccionar no corrige |
| Recuperar números guardados como texto | `to_numeric` | Lo inválido puede quedar como faltante |
| Unificar etiquetas | `strip`, `lower`, `replace` | No unir significados diferentes |
| Quitar una repetición accidental | `drop_duplicates` | No borrar observaciones válidas |
| Completar un faltante | `SimpleImputer` | No recupera el valor verdadero |
| Referencia 0–1 por columna | `MinMaxScaler` | Sensible a extremos |
| Media 0 y desviación 1 por columna | `StandardScaler` | No vuelve normal la distribución |
| Solo restar la media | `StandardScaler(with_std=False)` | No iguala escalas |
| Escalar con menor influencia de extremos | `RobustScaler` | No elimina los extremos |
| Comparar perfiles por fila | `normalize`, `Normalizer` | Se pierde el volumen |
| Representar una condición | `Binarizer` | Se pierde la cantidad |
| Representar categorías ordenadas | `OrdinalEncoder` | Puede imponer distancias artificiales |
| Representar categorías sin orden | `OneHotEncoder`, `get_dummies` | Puede generar muchas columnas |
| Convertir sí/no | `map` | Revisar etiquetas no contempladas |
| Codificar clases de la respuesta | `LabelEncoder` | Los códigos no son cantidades |

**Para recordar:** MinMax y StandardScaler transforman columnas; Normalizer transforma filas. Elegimos según lo que queremos conservar. No todas las variables necesitan escalado y no todos los modelos necesitan la misma representación.

**Mini práctica:** cambiá un número o una categoría en un ejemplo y anticipá el resultado antes de ejecutar. Explicá qué información se conservó y cuál se perdió.

Consulta: [documentación de preprocesamiento de scikit-learn](https://scikit-learn.org/stable/modules/preprocessing.html).